# Notebook 03 — Stage S: SimPy Discrete-Event Simulation

This notebook implements Stage S of the M-S-O pipeline:
1. Load `lambda_hat.csv` and `lp_assignment.json` from previous stages
2. Run 20 replications for each of the 3 dispatch policies (60 total)
   - Policy 1: Random Dispatch
   - Policy 2: Nearest-Feasible Greedy
   - Policy 3: LP-Optimized Dispatch
3. Save per-replication metrics for each policy

**Prerequisite:** Run notebooks 01 and 02 first.

**Estimated runtime: 5–15 minutes.**

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.utils.config_loader import load_config
from src.utils.distance import load_distance_matrix
from src.optimization.lp_formulation import LPDispatchSolver
from src.simulation.runner import SimulationRunner

print('Imports successful')

## 1. Load Inputs

In [ ]:
network_cfg = load_config(repo_root / 'config' / 'network.yaml')
sim_cfg = load_config(repo_root / 'config' / 'simulation.yaml')
sim_cfg['H'] = 12
sim_cfg['B'] = 3

# Load lambda_hat
lh_df = pd.read_csv(repo_root / 'data' / 'processed' / 'lambda_hat.csv', index_col=0)
lambda_hat = lh_df.values  # shape (12, 24)

# Load distance matrix and convert to flight times
d_km = load_distance_matrix(network_cfg)
speed_kmh = float(sim_cfg['drone']['speed_kmh'])

# Load LP assignment
x_sol = LPDispatchSolver.load(repo_root / 'data' / 'processed' / 'lp_assignment.json')

print(f'lambda_hat shape: {lambda_hat.shape}')
print(f'd_km shape: {d_km.shape}')
print(f'x_sol entries: {len(x_sol)}')
print(f'Replications: {sim_cfg["experiment"]["n_replications"]}')

## 2. Run Simulations (3 policies × 20 replications)

In [ ]:
n_reps = int(sim_cfg['experiment']['n_replications'])
seeds = list(range(1, n_reps + 1))  # CRN: seeds 1..20 shared

runner = SimulationRunner(
    sim_cfg=sim_cfg,
    lambda_hat=lambda_hat,
    d_km=d_km,
    x_sol=x_sol,
)

print(f'Running {n_reps} replications per policy with CRN seeds {seeds[0]}..{seeds[-1]}')
print()

results = runner.run_all_policies(n_replications=n_reps)

print('\nSimulation complete.')
for policy, df in results.items():
    print(f'  {policy}: {len(df)} replications')

## 3. Quick Summary

In [ ]:
metrics = ['FR', 'ADT', 'P95', 'ERR', 'AWT']
summary_rows = []
for policy, df in results.items():
    row = {'policy': policy}
    for m in metrics:
        if m in df.columns:
            row[f'{m}_mean'] = df[m].mean()
            row[f'{m}_std'] = df[m].std(ddof=1)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print('Quick summary (mean across 20 replications):')
print(summary_df[['policy'] + [f'{m}_mean' for m in metrics]].to_string(index=False))

## 4. Save Results

In [ ]:
out_dir = repo_root / 'data' / 'results'
SimulationRunner.save_results(results, out_dir)
print('Results saved:')
for policy in results:
    print(f'  data/results/sim_results_{policy}.csv')

## 5. Quick Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
policy_colors = {'random': '#e74c3c', 'greedy': '#3498db', 'lp': '#2ecc71'}
policy_labels = {'random': 'Random', 'greedy': 'Greedy', 'lp': 'LP-Optimized'}

for ax, metric in zip(axes, ['FR', 'ADT', 'ERR']):
    for i, policy in enumerate(['random', 'greedy', 'lp']):
        vals = results[policy][metric]
        ax.bar(i, vals.mean(), color=policy_colors[policy],
               label=policy_labels[policy], edgecolor='black')
        ax.errorbar(i, vals.mean(),
                    yerr=vals.std(ddof=1) / np.sqrt(len(vals)) * 2.093,
                    fmt='none', color='black', capsize=5)
    ax.set_xticks(range(3))
    ax.set_xticklabels([policy_labels[p] for p in ['random', 'greedy', 'lp']])
    ax.set_title(metric)

axes[0].set_ylabel('Fulfillment Rate (%)')
axes[1].set_ylabel('Avg Delivery Time (min)')
axes[2].set_ylabel('Expired Request Rate (%)')
fig.suptitle('Policy Comparison (mean ± 95% CI, 20 replications)', fontsize=12)
plt.tight_layout()
plt.show()

## Stage S Complete

**Outputs saved:**
- `data/results/sim_results_random.csv` — 20 replications, Random policy
- `data/results/sim_results_greedy.csv` — 20 replications, Greedy policy
- `data/results/sim_results_lp.csv` — 20 replications, LP-Optimized policy

**Next:** Run `notebook_04_evaluation.ipynb`